*Federated Learning under the lens of task arithmetic*

This notebook implements the project requirements in order:
1. Setup and Preliminaries
2. Gentle Starting (Centralized Baseline + Task Arithmetic)
3. The First FL Baseline
4. Simulate Heterogeneous Distributions
5. Task Arithmetic Techniques in FL
6. Extension (Guided or Custom)

*0. Setup - Mount Drive and Import Dependencies*

In [4]:
from google.colab import drive
drive.mount('/content/drive')

import sys, os
PROJECT_ROOT = "/content/drive/MyDrive/FL_PROJECT/Federated-Learning-Under-the-Lens-of-Task-Arithmetic"
print("Project exists?", os.path.exists(PROJECT_ROOT), PROJECT_ROOT)

if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)

print("sys.path OK")

# Standard imports
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm
import copy

# Project imports
from models.vit_dino import build_dino_vit as create_dino_and_define_train_mode
from data.datasets import get_cifar100, get_cifar100_transforms
from data.partition import make_dataset_loaders
from fl.dataloaders import build_federated_dataloaders
from train.eval import evaluate
from train.trainer import train_one_epoch_amp
from train.utils import make_scheduler, set_seed
from train.experiments import phase1_scheduler_sweep, phase2_lr_wd_grid

print("✓ All imports successful")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Project exists? True /content/drive/MyDrive/FL_PROJECT/Federated-Learning-Under-the-Lens-of-Task-Arithmetic
sys.path OK
✓ All imports successful


*1. PRELIMINARIES - Data Preparation and Validation Split*

In [6]:
print("\n" + "="*80)
print("PHASE 1: PRELIMINARIES - Data Preparation")
print("="*80)

model = create_dino_and_define_train_mode()

# Get transforms
transform_train, transform_test = get_cifar100_transforms()

# Download and split CIFAR-100 (train/val/test)
train_dataset, val_dataset, test_dataset = get_cifar100(
    transform_train,
    transform_test,
    val_ratio=0.1,
    root="./data",
    seed=42
)

# Create centralized dataloaders
train_loader_centr, val_loader_centr, test_loader_centr = make_dataset_loaders(
    train_dataset,
    val_dataset,
    test_dataset,
)

print(f"✓ Dataset split complete:")
print(f"  - Training samples: {len(train_dataset)}")
print(f"  - Validation samples: {len(val_dataset)}")
print(f"  - Test samples: {len(test_dataset)}")




PHASE 1: PRELIMINARIES - Data Preparation
✓ Dataset split complete:
  - Training samples: 45000
  - Validation samples: 5000
  - Test samples: 10000


*2. GENTLE STARTING - Centralized Baseline*

In [7]:
print("\n" + "="*80)
print("PHASE 2: CENTRALIZED BASELINE")
print("="*80)

print("\n--- 2.1: Scheduler Comparison ---")
# Compare different schedulers: cosine, step, exponential
phase1_results, phase1_histories = phase1_scheduler_sweep(
    train_loader_centr,
    val_loader_centr,
    test_loader_centr,
    epochs=15,
    lr=0.01,
    weight_decay=5e-4,
    img_size=160,
    seed=42
)

print("\n--- 2.2: Hyperparameter Grid Search ---")
# Grid search for best learning rate and weight decay
base_dir = PROJECT_ROOT
phase2_results = phase2_lr_wd_grid(
    train_loader_centr,
    val_loader_centr,
    test_loader_centr,
    scheduler_name="cosine",  # Best from phase 1
    epochs=15,
    lr_list=(0.003, 0.005, 0.01, 0.02),
    wd_list=(1e-4, 3e-4, 5e-4, 1e-3),
    img_size=160,
    seed=42,
    out_dir=base_dir,
    save_csv_path=f"{base_dir}/centralized_hyperparameters.csv",
    resume=True
)

print("\n✓ Centralized baseline complete")
print("Best hyperparameters saved to centralized_hyperparameters.csv")



PHASE 2: CENTRALIZED BASELINE

--- 2.1: Scheduler Comparison ---

=== PHASE 1 (scheduler sweep) ===


/content/drive/MyDrive/FL_PROJECT/Federated-Learning-Under-the-Lens-of-Task-Arithmetic/train/experiments.py:69: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=(device.startswith("cuda")))



[cosine] Epoch 001/15 ------------------------------


/content/drive/MyDrive/FL_PROJECT/Federated-Learning-Under-the-Lens-of-Task-Arithmetic/train/trainer.py:26: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
/usr/local/lib/python3.12/dist-packages/torch/amp/autocast_mode.py:270: UserWarning: User provided device_type of 'cuda', but CUDA is not available. Disabling
  warnings.warn(


  step 0001/352 | loss 5.0456 acc 0.0078 | 41.7s


KeyboardInterrupt: 

*TASK ARITHMETIC - Sparse Fine-Tuning Setup*

In [8]:
print("\n" + "="*80)
print("PHASE 3: TASK ARITHMETIC PREPARATION")
print("="*80)

# TODO: Implement SparseSGDM optimizer
# This extends standard SGDM with gradient masking capability

print("""
Task Arithmetic involves:
1. Computing Fisher Information Matrix (parameter sensitivity)
2. Creating gradient masks based on sensitivity scores
3. Using SparseSGDM optimizer that respects the masks
4. Multiple calibration rounds to refine the masks

Implementation references:
- Pruner code: https://github.com/iurada/talos-task-arithmetic/.../pruners.py
- Calibration: https://github.com/iurada/talos-task-arithmetic/.../prune_finetune.py
""")



PHASE 3: TASK ARITHMETIC PREPARATION

Task Arithmetic involves:
1. Computing Fisher Information Matrix (parameter sensitivity)
2. Creating gradient masks based on sensitivity scores
3. Using SparseSGDM optimizer that respects the masks
4. Multiple calibration rounds to refine the masks

Implementation references:
- Pruner code: https://github.com/iurada/talos-task-arithmetic/.../pruners.py
- Calibration: https://github.com/iurada/talos-task-arithmetic/.../prune_finetune.py



*THE FIRST FL BASELINE - FedAvg with IID Data*

In [9]:

print("\n" + "="*80)
print("PHASE 4: FEDAVG BASELINE (IID)")
print("="*80)

# Build IID federated dataloaders
client_loaders_iid, val_loader_fl, test_loader_fl, *_ = build_federated_dataloaders(
    train_transform=transform_train,
    test_transform=transform_test,
    K=100,           # 100 clients
    sharding="iid",  # IID distribution
    val_ratio=0.1,
    batch_size=64,
    seed=42
)

print(f"✓ Created {len(client_loaders_iid)} IID client dataloaders")

# TODO: Implement and run FedAvg
# Parameters:
# - K = 100 clients
# - C = 0.1 (10% client sampling per round)
# - J = 4 (local steps/epochs)
# - Proper number of rounds (based on convergence)

print("""
FedAvg Algorithm (McMahan et al., 2017):
1. Server initializes global model
2. For each round:
   a. Sample C*K clients randomly
   b. Send global model to selected clients
   c. Each client trains locally for J epochs
   d. Aggregate client models (weighted by dataset size)
   e. Update global model
3. Evaluate on validation/test set
""")


PHASE 4: FEDAVG BASELINE (IID)
✓ Created 100 IID client dataloaders

FedAvg Algorithm (McMahan et al., 2017):
1. Server initializes global model
2. For each round:
   a. Sample C*K clients randomly
   b. Send global model to selected clients
   c. Each client trains locally for J epochs
   d. Aggregate client models (weighted by dataset size)
   e. Update global model
3. Evaluate on validation/test set



*5. SIMULATE HETEROGENEOUS DISTRIBUTIONS*

In [ ]:

print("\n" + "="*80)
print("PHASE 5: HETEROGENEOUS DISTRIBUTIONS")
print("="*80)

# Parameters to test
nc_values = [1, 5, 10, 50]  # Classes per client
J_values = [4, 8, 16]       # Local steps
K = 100
C = 0.1

print(f"Testing combinations:")
print(f"  - Nc (classes per client): {nc_values}")
print(f"  - J (local steps): {J_values}")
print(f"  - Note: Rounds scale inversely with J (to keep total local steps constant)")

# Create non-IID dataloaders for each Nc value
non_iid_loaders = {}
for Nc in nc_values:
    client_loaders, val_loader, test_loader, *_ = build_federated_dataloaders(
        train_transform=transform_train,
        test_transform=transform_test,
        K=K,
        sharding="non_iid",
        Nc=Nc,
        val_ratio=0.1,
        batch_size=64,
        seed=42
    )
    non_iid_loaders[Nc] = (client_loaders, val_loader, test_loader)
    print(f"✓ Created Non-IID dataloaders for Nc={Nc}")

# TODO: Run experiments
# For each (Nc, J) combination:
# 1. Train FedAvg
# 2. Scale rounds: base_rounds * (4 / J)
# 3. Record best validation accuracy
# 4. Compare with IID baseline

print("""
Experiment Matrix:
- IID baseline: J ∈ {4, 8, 16}
- Non-IID: (Nc, J) ∈ {1,5,10,50} × {4,8,16}
- Total: 3 IID + 12 Non-IID = 15 experiments

Expected Findings:
1. Nc=1 (extreme heterogeneity) → Significant performance drop
2. Nc=50 (mild heterogeneity) → Close to IID performance
3. Larger J may amplify client drift effects
""")

*6. TASK ARITHMETIC TECHNIQUES IN FL*

In [ ]:
print("\n" + "="*80)
print("PHASE 6: TASK ARITHMETIC IN FEDERATED LEARNING")
print("="*80)

print("""
Sparse Fine-Tuning Process:
1. (Optional) Train/obtain frozen classifier locally
2. Calibrate gradient mask:
   - Compute parameter sensitivity (Fisher Information)
   - Identify least-sensitive parameters
   - Create binary mask (1 = update, 0 = freeze)
   - Multiple calibration rounds for refinement
3. Perform fine-tuning:
   - Use SparseSGDM optimizer with calibrated masks
   - Only update masked parameters

Experiments to run:
- Vary sparsity ratio (controlled by sensitivity threshold)
- Vary number of calibration rounds
- Compare with standard FedAvg
""")

# TODO: Implement sparse fine-tuning in FL
# 1. Implement Fisher Information computation
# 2. Implement gradient masking
# 3. Integrate with FedAvg
# 4. Run experiments

*7. EXTENSION - Choose One*

In [ ]:
print("\n" + "="*80)
print("PHASE 7: EXTENSION")
print("="*80)

print("""
OPTION A: Guided Extension
Compare different mask calibration rules:
1. Most-sensitive parameters (instead of least-sensitive)
2. Lowest-magnitude parameters
3. Highest-magnitude parameters
4. Random parameters
5. (Original) Least-sensitive parameters

OPTION B: Custom Extension
Examples:
- Novel task arithmetic techniques for FL
- Additional analysis of heterogeneity effects
- Architectural modifications for FL
- New aggregation strategies
- Communication-efficient adaptations

⚠️ Note: Quality of execution matters more than novelty!
""")

*RESULTS ANALYSIS AND VISUALIZATION*

In [ ]:
print("\n" + "="*80)
print("PHASE 8: RESULTS ANALYSIS")
print("="*80)

def analyze_results(results_dict):
    """
    Analyze and visualize experimental results
    """
    # Create comprehensive plots
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))

    # Plot 1: IID vs Non-IID comparison
    # Plot 2: Effect of local steps (J)
    # Plot 3: Effect of heterogeneity (Nc)
    # Plot 4: Task arithmetic improvements

    plt.tight_layout()
    plt.savefig('results_summary.png', dpi=300)
    plt.show()

    # Generate summary statistics
    summary = pd.DataFrame(results_dict)
    print("\nSummary Statistics:")
    print(summary.describe())

    return summary

print("""
Analysis should include:
1. Comparison plots (IID vs Non-IID)
2. Performance degradation metrics
3. Effect of local steps on convergence
4. Task arithmetic benefits quantification
5. Extension results and insights

All plots and tables should be publication-ready for the report.
""")


*UTILITIES AND HELPER FUNCTIONS*

In [ ]:
def save_checkpoint(model, optimizer, epoch, metrics, path):
    """Save training checkpoint"""
    torch.save({
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'metrics': metrics
    }, path)
    print(f"✓ Checkpoint saved: {path}")

def load_checkpoint(path, model, optimizer=None):
    """Load training checkpoint"""
    ckpt = torch.load(path, map_location='cpu')
    model.load_state_dict(ckpt['model_state_dict'])
    if optimizer:
        optimizer.load_state_dict(ckpt['optimizer_state_dict'])
    print(f"✓ Checkpoint loaded: {path}")
    return ckpt['epoch'], ckpt['metrics']